In [2]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         PDF EXTRACTOOOOOOOR         #
#######################################

# Define target website
URL = "https://www.clc.gov.sg/research-publications/publications/commentaries-reports"
domain = "https://www.clc.gov.sg"

# Define variables
files = {}
folder_path = "docs"
hrefSchema = {}
docsFolderPath = "/publications/contributions/"

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL)
  soup = BeautifulSoup(page.content, "html.parser")

  # 1. Loop through all <a> tags found in HTML
  # 2. Attempt to detect '/docs/' for each href link in <a> tag
  # 3. Format file name
  # 4. Build dictionary with extracted data
  # 5. Initiate downloadFiles() function once 1-4 is completed
  for i in soup.find_all(['a']):
    href = i.get('href')
    # print(href)
    if href and '/docs/' in i['href'] or href and 'go.gov.sg' in i['href']:

      # Build dictionary for easier search
      # f-strings -> To embed variables directly to strings
      # fileName -> Dictionary Comprehension
      # Count -> Unique identifier for access
     
      files[count] = {"oriFileName": i.text, "pdfName": i['href'][:i['href'].find('?')].split("/")[-1], "link": i['href'], "pureLink": i['href'][:i['href'].find('?')], "downloadLink": i['href'] if 'go.gov.sg' in i['href'] else domain + i['href'] if domain not in i['href'] else i['href']}
      r = requests.head(files[count]["downloadLink"], allow_redirects=True)
      files[count]["downloadLink"] = r.url.split('%')[0]
      files[count]["pdfName"] = r.url[:r.url.find('?')].split("/")[-1] if '?' in r.url else r.url.split("/")[-1]
      files[count]["pdfName"] = files[count]["pdfName"].split('%')[0]

      print(files[count]["downloadLink"], files[count]["pdfName"])

      count += 1
      # print(domain + i['href'] if domain not in i['href'] else i['href'])
  downloadFiles()

def downloadFiles():
  # Create 'docs' folder if it doesnt exists on Google Colab workspace
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)

  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # 1. Loops through each dictionary item
  # 2. Retrieve a file's download link
  # 3. Define the file's intended file path and name
  # 4. Copy the file and store in its intended file path and name
  # 5. Zip the folder once 1-4 is completed
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['downloadLink']

        # Define file path and name
        file_path = os.path.join(f"{folder_path}", f"{files[index]['pdfName']}")
        
        # Copy file from download link
        urllib.request.urlretrieve(url, file_path)

    except Exception as e:
        pass
        
  # Zip files
  # zip_file_path = shutil.make_archive(folder_path, 'zip', folder_path)

  # 1. Loops through each dictionary item
  # 2. Calculate file size in bytes
  # 3. Calculate file size based on bytes to use between kB/MB/GB
  # 4. Build json schema for hyperlinks
  # 5. Print json schema
  for index, value in enumerate(files):
    try:
        # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"docs/{files[index]['pdfName']}")/1024))
    
        # Calculate and define file size
        # Removed temporarily - str(int(fileSize)/1000000) + ' GB'
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'
    
        # Build with hardcoded json schema
        # f-strings -> To embed variables directly to strings
        hrefSchema = {
              "type": "text",
              "marks": [
                {
                  "type": "link",
                  "attrs": {
                    "href": f"/files{docsFolderPath if True else None}{files[index]['pdfName']}"
                    # "href": f"{files[index]['pureLink']}"
                    }
                }
              ],
              # Original File Name is used as we want to display Circular 1 instead of circular-1
              "text": f"{files[index]['oriFileName']} [{'DOCX' if '.docx' in files[index]['downloadLink'] else 'DOC' if '.doc' in files[index]['downloadLink'] else 'XLXS' if '.xlxs' in files[index]['downloadLink'] else 'XLS' if '.xls' in files[index]['downloadLink'] else 'ZIP' if '.zip' in files[index]['downloadLink'] else 'PDF'}, {fileSize}]"
            }
            
        print(json.dumps(hrefSchema))
    except Exception as e:
      pass

getDictionary(URL)

df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("urban-solutions.csv", index=False)

https://www.clc.gov.sg/docs/default-source/commentaries/future-senior-housing-commentary.pdf future-senior-housing-commentary.pdf
https://www.clc.gov.sg/docs/default-source/commentaries/resilient-sydney-platform-commentary.pdf resilient-sydney-platform-commentary.pdf
https://www.clc.gov.sg/docs/default-source/commentaries/clc_commentary_health-districts.pdf clc_commentary_health-districts.pdf
https://www.clc.gov.sg/docs/default-source/commentaries/bc_2018_02_master_developer_projects_in_singapore.pdf?sfvrsn=760d8789_0 bc_2018_02_master_developer_projects_in_singapore.pdf
https://www.clc.gov.sg/docs/default-source/commentaries/a-softer-approach-to-managing-waste.pdf?sfvrsn=a0d77be6_0 a-softer-approach-to-managing-waste.pdf
https://www.clc.gov.sg/docs/default-source/commentaries/firstthingsfirst-(1)bd1bd15d7fa04b308b9111888966fb8a.pdf?sfvrsn=7e773c30_0 firstthingsfirst-(1)bd1bd15d7fa04b308b9111888966fb8a.pdf
https://www.clc.gov.sg/docs/default-source/commentaries/reinventer-paris-(reinve

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math

#######################################
#         Image EXTRACTOOOOOOOR       #
#######################################

# Define target website
URL = "https://www.clc.gov.sg/docs/default-source/contributions/the-asean-issue-14---viewpoint-hugh-lim.pdf?sfvrsn=2b1e131_2"
domain = "https://www.clc.gov.sg"

# Define variables
files = {}
folder_path = "images"
hrefSchema = {}
docsFolderPath = "/images/"

# Define the folder path
file_name = 'docs'
file_name_zip = file_name+'.zip'

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
        os.remove(file_name_zip)
except:
  pass

def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL)
  soup = BeautifulSoup(page.content, "html.parser")

  # 1. Loop through all <a> tags found in HTML
  # 2. Attempt to detect '/docs/' for each href link in <a> tag
  # 3. Format file name
  # 4. Build dictionary with extracted data
  # 5. Initiate downloadFiles() function once 1-4 is completed
  for i in soup.find_all(['img']):
    try:
        # Remove and replace spaces and with dashes + lowercaps for file name (e.g. Circular 1 -> circular-1)
    
        # Build dictionary for easier search
        # f-strings -> To embed variables directly to strings
        # fileName -> Dictionary Comprehension
        # Count -> Unique identifier for access
        files[count] = {"oriFileName": i['alt'], "fileName": i['src'][:i['src'].find('?')].split("/")[-1], "downloadLink": domain + i['src']}
        count += 1
        
    except Exception as e:
        # print(e)
        pass

  downloadFiles()

def downloadFiles():
  # Create 'docs' folder if it doesnt exists on Google Colab workspace
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)

  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # 1. Loops through each dictionary item
  # 2. Retrieve an image's download link
  # 3. Define the image's intended file path and name
  # 4. Copy the image and store in its intended file path and name
  # 5. Zip the folder once 1-4 is completed
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['downloadLink']

        # Define file path and name
        file_path = os.path.join(f"{folder_path}", f"{files[index]['fileName']}")
        
        # Copy file from download link
        urllib.request.urlretrieve(url, file_path)

        hrefSchema = f"{docsFolderPath}{files[index]['fileName']}"
                
    except Exception as e:
        pass
        
  # Zip files
  # zip_file_path = shutil.make_archive(folder_path, 'zip', folder_path)
  print("Done")

getDictionary(URL)

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math

#######################################
#     Link Extractooooor (Website)    #
#######################################

# Define target website
url = "https://www.clc.gov.sg/events/lectures"
domain = "https://www.clc.gov.sg"

def getDictionary(link):
  count = 0

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(link)
  soup = BeautifulSoup(page.content, "html.parser")
  soup = soup.find("div", class_="sf_colsIn sf_2cols_1in_75")
  # print(soup)
  # 1. Loop through all <a> tags found in HTML
  # 2. Build dictionary with extracted data
  # 3. Initiate downloadFiles() function once 1-4 is completed
  for i in soup.find_all(['a']):
    # href = i.get('href')
    print(i)
    try:
        # Remove and replace spaces and with dashes + lowercaps for file name (e.g. Circular 1 -> circular-1)
        replacedString = i['alt'].replace(".", "-")
        replacedString = replacedString.replace(" ", "-")
        replacedString = replacedString.replace(",", "-")
        replacedString = replacedString.replace("(", "-")
        replacedString = replacedString.replace("?", "-")
        replacedString = replacedString.replace(":", "-")
        replacedString = replacedString.replace("&", "-")
        replacedString = replacedString.replace("|", "-")
        replacedString = replacedString.replace("%", "-")
        replacedString = replacedString.replace(")", "-")
        replacedString = replacedString.replace('"', "-")
        replacedString = replacedString.replace("、", "-")
        replacedString = replacedString.replace("/", "-")
        replacedString = replacedString.replace("\\", "-")
        replacedString = replacedString.replace("/", "-").lower()
    
        # Build dictionary for easier search
        # f-strings -> To embed variables directly to strings
        # fileName -> Dictionary Comprehension
        # Count -> Unique identifier for access
        files[count] = {"oriFileName": i['alt'], "fileName": replacedString if len(replacedString) != 0 else f"temp{count}", "downloadLink": domain + i['src']}
        count += 1
    except Exception as e:
        # print(e)
        pass

getDictionary(url)

In [ ]:
import pandas as pd
import json
import urllib.request
import requests
from bs4 import BeautifulSoup

########################################
#       Link Extractoooor (CSV)        #
########################################

url = "https://www.agc.gov.sg/newsroom/media-releases/newsitem/"

df = pd.read_csv('agc.csv')
df_copy = df.copy()
df_copy['Category'] = None

for index, value in enumerate(df_copy['Title']):
    page = requests.get(f"{url}{df_copy['Original page name'][index]}")
    soup = BeautifulSoup(page.content, "html.parser")
    soup = soup.find("article", class_="news-single--content mb60")
    print(soup)
    df_copy.loc[index, "Category"] = soup.find('h1').text

df_copy.to_csv('agc.csv', index=False, header=True)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup
import requests
import re
import math

########################################
#        HTML Extractoooor (Link)      #
########################################

domain = "https://www.clc.gov.sg"
URL = "https://www.clc.gov.sg/research-publications/publications/better-cities/GetDataView//"
count = 0
data = {}

# Get links in page
for number in range (0,5):
    tempLinks = []
    page  = requests.post(URL, data = {'DivName':f'articleList202{number}', 'Year':f'202{number}'})
    soup = BeautifulSoup(page.content, "html.parser")
    
    for i in soup.find_all(['a']):
        href = i.get('href')
        if "/" in href and href not in tempLinks:
            tempLinks.append(href)
    
    for i in tempLinks:
        data[count] = {"Year": f"202{number}", "url": "https://www.clc.gov.sg" + i, "path": i}
        count+=1

for i in range(0, len(data)):
    page = requests.get(data[i]["url"])
    soup = BeautifulSoup(page.content, "html.parser")
    
    content = soup.find("div", class_="sf_2cols_1in_75")
    content1 = content.find_all("div", class_="row")
    data[i]["html"] = content1

df = pd.DataFrame.from_dict(data, orient='index')
df.to_csv("better-cities.csv", index=False)

In [256]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# Declare dataset as global variable 
df = pd.read_csv('public-consultation.csv')
df_copy = df.copy()
pageData = {}

# df_copy["Title"] = df_copy["URL"].apply(lambda url: url.split("/")[-1])

for index, value in enumerate(df_copy['URL']):
    page = requests.get(df_copy['URL'][index])
    soup = BeautifulSoup(page.content, "html.parser")

    # title = soup.find("div", class_="sfContentBlock sf-Long-text a-rich-text").find("h1").text
    title = value.split("/")[-1].replace("-", " ").upper()  
    header = soup.find("div", class_="description-group")
    date = header.find("dl", class_="description-list class").find("dd").text.split("-")[0][:-1]
    content = soup.find("div", class_="accordion accordion--consultation")
    try:
        pdf = soup.find_all("div", class_="mt-5")[1]
    except Exception as e:
        pdf = "No PDF"
    pageData[index] = {"Title": title, "Date": date, "URL": value, "header": header, "content": content, "pdf": pdf}

export = pd.DataFrame.from_dict(pageData, orient='index')
export.to_csv("public-consultation-processed.csv", index=False)

In [160]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#        HTML Extractoooor (CSV)       #
########################################

# Declare dataset
df = pd.read_csv('reach-automation.csv')
df_copy = df.copy()

# List to store HTML code
pageData = []

# Loop through each URL in the csv file
for index, value in enumerate(df_copy['Site link']):
    # Fetch site's HTML
    page = requests.get(df_copy['Site link'][index])
    soup = BeautifulSoup(page.content, "html.parser")

    # Pinpoint the exact "Container" that hosts the content we want
    content = soup.find_all("div", class_="sfContentBlock sf-Long-text a-rich-text")[2]
    pageData.append(content.prettify())

# Insert pageData list to a new column called "HTML"
df_copy["HTML"] = pageData

# Export the dataframe into csv file
df_copy.to_csv("automation-processed.csv", index=False)

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (API, CLC)     #
########################################

# URL of the API
url = "https://www.clc.gov.sg/research-publications/publications/digital-library/Search/"
domain = "https://www.clc.gov.sg"
pageData = {}

counter = 1
pageCounter = 0
while counter < 19:
    form_data = {
        "page": f"{counter}"    
    }
    
    response = requests.post(url, data=form_data)
    response.raise_for_status()  # Check if the request was successful
    soup = BeautifulSoup(response.text, "html.parser")
    content = soup.find_all("div", class_ = "digital-library-item-result")
    
    for i, v in enumerate(content):
        try:
            title = v.find("h4", class_ = "title")
            type = v.find("span", class_ = "publication-type")
            publication = v.find("span", class_ = "publication-type-hidden")
            blurb = v.find("p", class_ = "blurb")
            pageData[pageCounter] = {"Title": title.text, "URL": domain + title.find("a").get('href'), "Publication type": type.text, "Category": publication.text, "Subtitle": blurb.text}
        except Exception as e:
            pageData[pageCounter] = {"Title": title.text, "URL": domain + title.find("a").get('href'), "Publication type": None, "Category": None, "Subtitle": blurb.text}
            print(title.text)

        print(pageCounter)
        pageCounter += 1
    counter += 1

for i, v in enumerate(pageData):
    response = requests.post(pageData[i]["URL"])
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    content = soup.find_all("div", class_ = "sf_colsIn sf_2cols_1in_67")
    pageData[i]["HTML"] = content

(pd.DataFrame.from_dict(data=pageData, orient='index')
   .to_csv('clc-digital-library.csv', header=True))

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup
import re

########################################
#     HTML Extractoooor (API, MUIS)    #
########################################

# URL of the API
url = "https://www.muis.gov.sg/officeofthemufti/Khutbah"
yearList = ["2024", "2023", "2022", "2021", "2020", "2019", "2018", "2017", "2016"]
languageList = ["493dd241-c9dc-48e7-af8f-34aa2a07ce10", "f3dba80c-5b3b-4866-8dec-b9ac0fb02b59", "57fc2908-8a6b-41fa-8ed2-f93771dc1bcb"]

prayerDict = {}


for year in yearList:
    for language in languageList:
        titleList = []
        try:
            request_body = {
                "scAction": "GetKhubatArticles",
                "scController": "PageContent",
                "searchData[GUID]": f"{language}",
                "searchData[PageSize]": "1000",
                "searchData[Page]": "1",
                "searchData[Year]": f"{year}"
            }
            headers = {'Content-Type': 'application/x-www-form-urlencoded'}
            response = requests.post(url, data=request_body, headers=headers)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            date = soup.find_all("p", class_="italic")
            dateList = []
            contentListEng = []
            
            for i in date:
                dateFormatted = i.text.replace("\r", "").replace("\n", "")
                dateList.append(dateFormatted[:list(re.finditer(r'\d+', dateFormatted))[-1].end()])
            
            content = soup.find_all("div", class_="row")
            title = soup.find_all("p", class_="strong")
            
            for i in title:
                titleList.append(i.text)
            
            while True:
                if len(content) == 0:
                    break
                contentListEng.append(str(content[0]) + str(content[1]) if len(content) % 2 == 0 else str(content[0]))
                try:
                    del content[0]
                except Exception as e:
                    pass
                try:
                    del content[0]
                except Exception as e:
                    pass
            
            for i, v in enumerate(dateList):
                if v not in prayerDict:
                    prayerDict[v] = {}
                try:
                    prayerDict[v]["Title"] = titleList[i]
                    prayerDict[v][language] = contentListEng[i]
                except Exception as e:
                    prayerDict[v][language] = ""
                    
        except Exception as e:
            print(year, language)
            continue
                    
print(prayerDict)

(pd.DataFrame.from_dict(data=prayerDict, orient='index')
   .to_csv('muis.csv', header=True))